In [1]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

c:\Users\goran\AppData\Local\Programs\Python\Python313\Lib\ssl.py:524: UserWarning: Bad certificate in Windows certificate store: not enough data: cadata does not contain a certificate (_ssl.c:4218)
  warnings.warn(f"Bad certificate in Windows certificate store: {exc!s}")


In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [3]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [4]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [5]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [6]:
import json

with open("lesson11-dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [7]:
print(json.dumps(results, indent=2))

[
  {
    "output": "Looking at the AWS ARN format, I need to extract the service name from the ARN string.\n\nAWS ARN format is:\n```\narn:partition:service:region:account-id:resource\n```\n\nThe service name is the 3rd component (index 2) when splitting by colons.\n\nHere's the solution:\n\n```python\ndef extract_service_from_arn(arn):\n    \"\"\"\n    Extracts the service name from an AWS ARN string.\n    \n    Args:\n        arn (str): AWS ARN string (e.g., 'arn:aws:s3:::my-bucket')\n    \n    Returns:\n        str: Service name (e.g., 's3')\n    \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        raise ValueError(\"ARN must be a non-empty string\")\n    \n    parts = arn.split(':')\n    \n    # Valid ARN should have at least 6 parts\n    if len(parts) < 6:\n        raise ValueError(f\"Invalid ARN format: {arn}\")\n    \n    # Check if it starts with 'arn'\n    if parts[0] != 'arn':\n        raise ValueEr